In [ ]:
import os
import gc
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../")))
from srlib.dataset.loading import load_edsr_dataset
from srlib.dataset.visualization import plot_patch_pairs
from srlib.deep_learning.edsr import EDSR
from srlib.constants import (
    CLASS_LABELS_PATH,
    EDSR_PATCH_SIZE,
    EDSR_SCALE_FACTOR,
    EDSR_STRIDE,
    HR_ROOT,
    LR_ROOT,
)

In [ ]:
X_train, Y_train, X_val, Y_val, X_test, Y_test = load_edsr_dataset(
    HR_ROOT,
    LR_ROOT,
    CLASS_LABELS_PATH,
    patch_size=EDSR_PATCH_SIZE,
    stride=EDSR_STRIDE,
    scale_factor=EDSR_SCALE_FACTOR,
)

In [ ]:
# What the model is actually fed: the LR patch and the HR patch it must
# reconstruct, whose side is EDSR_SCALE_FACTOR times larger.
_ = plot_patch_pairs(X_train, Y_train, count=3, seed=0)

In [ ]:
model = EDSR()

model.setup_model(
    scale_factor=EDSR_SCALE_FACTOR, 
    num_res_blocks=16, 
    num_filters=64, 
    learning_rate=5e-5
)

In [ ]:
# Train EDSR and capture callbacks for metrics
history, time_cb, mem_cb = model.fit(
    X_train, Y_train, X_val, Y_val, 
    batch_size=16, 
    epochs=300
)

In [ ]:
del X_train, Y_train, X_val, Y_val  # Free memory
gc.collect()  # Force garbage collection to free memory

In [ ]:
# Evaluates the run, saves the checkpoint and the metrics under one
# timestamp, so the reporting notebooks resolve a single run.
timestamp, run_dir, metrics = model.evaluate_and_save(
    X_test, Y_test, history, time_cb, mem_cb,
)